In [1]:
# Importing necessary libraries
import pandas as pd

In [2]:
# Loading Dataset
movies = pd.read_csv(filepath_or_buffer="../data/movies.csv").loc[:, ["original_title", "overview"]]
movies.head()

,original_title,overview
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


In [3]:
# Preprocessing 1 - Lowercasing
movies['overview'] = movies['overview'].str.lower()
movies.head()

,original_title,overview
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


In [4]:
# Preprocessing 2 - Removing HTML Tags
import re

# Incase there are null values
movies.dropna(inplace=True)

def remove_html_tags(text):
    pattern = re.compile("<.*?>")
    return pattern.sub(r'', text)

movies['overview'] = movies['overview'].apply(remove_html_tags)
movies.sample(5)

,original_title,overview
1889,He Got Game,a basketball player's father must try to convi...
2059,Molière,"molière, a down-and-out actor-cum-playwright u..."
1827,Teenage Mutant Ninja Turtles II: The Secret of...,the turtles and the shredder battle once again...
79,Iron Man 2,with the world now aware of his dual life as t...
3611,A Farewell to Arms,british nurse catherine barkley (helen hayes) ...


In [5]:
# Preprocessing 3 - Removing Links
def removing_links(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')

    # This pattern is useful to filterout all types of links
    '''
        text1 = 'Check out my notebook https://www.kaggle.com/campusx/notebook8223fc1abb'
        text2 = 'Check out my notebook http://www.kaggle.com/campusx/notebook8223fc1abb'
        text3 = 'Google search here www.google.com'
        text4 = 'For notebook click https://www.kaggle.com/campusx/notebook8223fc1abb to search check www.google.com'
    '''

    return pattern.sub(r'', text)

movies['overview'] = movies['overview'].apply(removing_links)
movies.sample(5)

,original_title,overview
3346,Jumping the Broom,two very different families converge on martha...
3276,It's Kind of a Funny Story,a clinically depressed teenager gets a new sta...
2643,Возвращение,a story of two russian boys whose father sudde...
2626,Idle Hands,anton is a cheerful but exceedingly non-ambiti...
366,Hollow Man,"cocky researcher, sebastian caine is working o..."


In [6]:
from urlextract import URLExtract # Incase you have'nt installed - !pip install urlextract

extractor = URLExtract()
urls = extractor.find_urls("Let's have URL www.stackoverflow.com as an example.")
print(urls) # prints: ['stackoverflow.com']

['www.stackoverflow.com']


In [7]:
# Preprocessing 4 - Remove Punctuations
import string
exclude = string.punctuation
print(exclude)

def remove_punctuation(overview):
    return "".join([(t if t not in exclude else "") for t in overview])
    # return text.translate(str.maketrans('', '', exclude)) - maketrans is a mapping function

movies['overview'] = movies['overview'].apply(remove_punctuation)
movies.sample(5)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


,original_title,overview
4612,Old Joy,two old pals reunite for a camping trip in ore...
2907,Pet Sematary,dr louis creeds family moves into the country ...
4314,Crowsnest,in late summer of 2011 five young friends on a...
3391,Dom Hemingway,after spending 12 years in prison for keeping ...
4375,Blue Car,gifted 18yearold meg has been abandoned by her...


In [8]:
# Preprocessing 5 - Chat Words Treatment
import json

with open(file="../data/chat_words_dictonary.json") as f:
    chat_words = json.load(f)

for index, (key, value) in enumerate(chat_words.items()):
    if index == 10: break
    print(f"{key} : {value}")

atm : at this moment
brb : be right back
lol : laughing out loud
btw : by the way
g2g : got to go
idk : I don't know
ikr : I know, right
np : no problem
omg : oh my god
rofl : rolling on floor laughing


In [9]:
def chat_words_treatment(text):
    return " ".join(list(map(lambda t: t if t not in chat_words else chat_words[t], text.split(" "))))

movies['overview'] = movies['overview'].apply(chat_words_treatment)
movies.sample(5)

,original_title,overview
2431,My Life in Ruins,a greek tour guide named georgia attempts to r...
3689,All Hat,an excon returns to his rural ontario roots an...
3226,Namastey London,indianborn manmohan malhotra decided to reloca...
1135,Lord of War,yuri orlov is a globetrotting arms dealer and ...
1242,The Reaping,katherine morrissey a former christian mission...


In [ ]:
# Preprocessing 6 - Spelling Correction
from textblob import TextBlob

def spelling_correction(text):

    textBlb = TextBlob(text) # Creating TextBlob object
    return textBlb.correct().string # Correcting spelling

spelling_correction("Hellow myi nmame ins dexter morgan")
# Note: Even for small datasets it takes lot of time

In [ ]:
# Preprocessing 7 - Removing Stop-Words
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stopwords = stopwords.words('english')

def remove_stopwords(text):
    result = []
    for word in text.split(" "):
        if word not in stopwords:
            result.append(word)
    return " ".join(result)

movies['Description'] = movies['Description'].apply(remove_stopwords)

In [15]:
# Preprocessing 8 - Removing Emojis
def remove_emoji(text):
    emoji_pattern = re.compile("["
            u"\U0001F600-\U0001F64F"  # emoticons
            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # transport & map symbols
            u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
            u"\U00002702-\U000027B0"
            u"\U000024C2-\U0001F251"
        "]+", 
        flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

remove_emoji("Loved the movie. It was 😘😘")

# movies['Description'] = movies['Description'].apply(remove_emoji)

'Loved the movie. It was '

In [ ]:
# Preprocessing 8' - Replacing emojis with there english representation
import emoji

print(emoji.demojize('Python is 🔥'))
print(emoji.demojize('Loved the movie. It was 😘'))
print(emoji.demojize('Loved the movie. It was 😂'))

Python is :fire:
Loved the movie. It was :face_blowing_a_kiss:
Loved the movie. It was :face_with_tears_of_joy:


In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize
tokens = word_tokenize("".join(movies['Description']))

In [ ]:
tokens[:5]

['imprisoned', '1940s', 'double', 'murder', 'wife']

In [ ]:
movies.loc[0, 'Description']

'imprisoned 1940s double murder wife lover upstanding banker andy dufresne begins new life shawshank prison puts accounting skills work amoral warden long stretch prison dufresne comes admired inmates including older prisoner named red integrity unquenchable sense hope'

In [ ]:
sentences_tokens = sent_tokenize("".join(movies['Description']))
# Since we removed all the puncuations we got only one token

In [ ]:
sent_tokenize("Hello my name is Dexter. I live at Kolhapur")

['Hello my name is Dexter.', 'I live at Kolhapur']

In [ ]:
import spacy

# Creating blank language object then
# tokenizing words of the sentence
nlp = spacy.blank("en")

doc = nlp("am am are are")
doc

am am are are

In [ ]:
tokens = sent_tokenize("$10 15km 15-20 U.S.")
doc = nlp("$10 15km 15-20 U.S.")

print(tokens)

['$10 15km 15-20 U.S.']


In [ ]:
doc

$10 15km 15-20 U.S.

## Lemmatization

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download WordNet and punkt if you haven't already
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
# Initialize the lemmatizer
wordnet_lemmatizer = WordNetLemmatizer()

# Sample sentence
sentence = "He was running and eating at the same time. He has a bad habit of swimming after playing long hours in the Sun."

# Tokenize the sentence into words
words = word_tokenize(sentence)

# Lemmatize each word in the sentence with 'v' as the POS tag for verbs
lemmatized_words = [wordnet_lemmatizer.lemmatize(word, pos='v') for word in words]

# Join the lemmatized words back into a sentence
lemmatized_sentence = ' '.join(lemmatized_words)

print(lemmatized_sentence)

He be run and eat at the same time . He have a bad habit of swim after play long hours in the Sun .


# Text Representation

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(lowercase = True, stop_words = 'english', analyzer = 'word', ngram_range = (1, 3), binary = False)
bow = cv.fit_transform(movies['Description'])

In [ ]:
cv.vocabulary_

{'imprisoned': 191271,
 '1940s': 1389,
 'double': 108876,
 'murder': 258961,
 'wife': 422043,
 'lover': 233240,
 'upstanding': 407478,
 'banker': 30302,
 'andy': 16886,
 'dufresne': 112313,
 'begins': 34362,
 'new': 265888,
 'life': 221885,
 'shawshank': 344919,
 'prison': 299946,
 'puts': 305621,
 'accounting': 6273,
 'skills': 350555,
 'work': 427250,
 'amoral': 16173,
 'warden': 416720,
 'long': 229465,
 'stretch': 367680,
 'comes': 71461,
 'admired': 8057,
 'inmates': 194183,
 'including': 191776,
 'older': 274225,
 'prisoner': 300287,
 'named': 262081,
 'red': 313608,
 'integrity': 195676,
 'unquenchable': 406258,
 'sense': 340372,
 'hope': 183544,
 'imprisoned 1940s': 191274,
 '1940s double': 1392,
 'double murder': 108913,
 'murder wife': 259355,
 'wife lover': 422494,
 'lover upstanding': 233339,
 'upstanding banker': 407479,
 'banker andy': 30305,
 'andy dufresne': 16898,
 'dufresne begins': 112314,
 'begins new': 34645,
 'new life': 266617,
 'life shawshank': 223525,
 'shawsh

In [ ]:
len(cv.vocabulary_)

29874

In [ ]:
bow[0].toarray()

array([[0, 0, 0, ..., 0, 0, 0]])

In [ ]:
from sklearn.feature_extraction.text import TfimoviesVectorizer

tfimovies = TfimoviesVectorizer()
encoded = tfimovies.fit_transform(movies['Description'])

In [ ]:
encoded[0].toarray()

array([[0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
print(tfimovies.imovies_)

[9.08343148 9.48889659 8.3902843  ... 9.48889659 9.48889659 9.08343148]


array(['00', '006', '007', ..., 'éric', 'öztürk', 'ʻohana'], dtype=object)

# Deep Learning Approaches

In [ ]:
!pip install gensim

In [ ]:
from gensim.utils import simple_preprocess

In [ ]:
simple_preprocess("Hello My name is Harry")

['hello', 'my', 'name', 'is', 'harry']

In [ ]:
store = []
movies['Description'].apply(lambda x: store.append(simple_preprocess(x)))

,Description
0,None
1,None
2,None
3,None
4,None
...,...
9715,None
9716,None
9717,None
9718,None


In [ ]:
len(store)

9720

In [ ]:
import gensim
model = gensim.models.Word2Vec(
    window = 10,
    min_count = 2,
    vector_size = 100
)

In [ ]:
model.build_vocab(store)

In [ ]:
model.epochs

5

In [ ]:
model.train(store, total_examples = model.corpus_count, epochs = model.epochs)

(1177118, 1261000)

In [ ]:
model.wv.most_similar('boy')

[('meets', 0.9993821382522583),
 ('son', 0.9993306994438171),
 ('mother', 0.9992178678512573),
 ('married', 0.9991076588630676),
 ('couple', 0.9990133047103882),
 ('beautiful', 0.9989761710166931),
 ('whose', 0.9989615678787231),
 ('father', 0.9989044070243835),
 ('daughter', 0.9989011883735657),
 ('teenage', 0.9988861083984375)]

In [ ]:
model.wv.doesnt_match(['boy', 'girl', 'man', 'women', 'merry'])

'merry'

In [ ]:
model.wv['king']

array([-4.7805241e-01,  8.9421040e-01,  5.9682089e-01,  4.0302407e-02,
       -1.8417187e-01, -1.0727488e+00,  3.2542974e-01,  1.6779027e+00,
       -3.6795244e-01, -2.4228179e-01, -3.5064438e-01, -1.4925445e+00,
       -1.0779732e-02,  3.4555501e-01,  5.9883475e-01, -1.2025588e+00,
        6.4149626e-02, -1.2701110e+00, -3.3447571e-02, -1.7758778e+00,
        4.0661937e-01,  5.2599221e-01,  5.1979232e-01, -2.7209094e-01,
       -3.7241276e-02, -1.4135563e-02, -6.5465808e-01, -4.3948710e-01,
       -9.2859024e-01, -1.2027569e-01,  5.4470801e-01,  1.1613366e-01,
        1.7044647e-03, -5.4109949e-01, -7.6100066e-02,  8.7756443e-01,
        3.2019526e-01, -6.0449409e-01, -3.3756110e-01, -1.6588789e+00,
       -4.8590802e-02, -7.6649451e-01, -4.9502033e-01,  1.7559346e-02,
        5.0756824e-01, -3.2639140e-01, -7.8039438e-01, -5.7014484e-02,
       -1.4682734e-01,  8.7216234e-01, -1.2324012e-01, -6.3025528e-01,
       -7.1349901e-01, -2.1083152e-01, -7.2468585e-01,  7.2306651e-04,
      

In [ ]:
model.wv.similarity('bottle','can')

0.99186695

In [ ]:
model.wv.get_normed_vectors() # Returns the vector representation of all the words

array([[-0.06355576,  0.13680956,  0.09171266, ..., -0.15839307,
         0.03032753,  0.01129388],
       [-0.08647829,  0.1175487 ,  0.10407068, ..., -0.14106596,
        -0.01215109, -0.0006144 ],
       [-0.05690471,  0.15416141,  0.07277973, ..., -0.15262741,
         0.0419447 ,  0.00231048],
       ...,
       [-0.12808035,  0.18103282,  0.08616749, ..., -0.18451056,
        -0.01988638,  0.00138477],
       [-0.12168589,  0.08864621,  0.03618131, ..., -0.11659244,
        -0.04106048,  0.08991076],
       [-0.09016749,  0.1173208 ,  0.0997637 , ..., -0.14454001,
         0.05217974, -0.01003663]], dtype=float32)

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components = 3)

In [ ]:
pca.fit_transform(model.wv.get_normed_vectors())

array([[-0.03644544, -0.00485121, -0.03764542],
       [-0.00977886, -0.09970655, -0.10137251],
       [-0.02839023, -0.01952115, -0.08144549],
       ...,
       [ 0.03777266,  0.03631732,  0.04492512],
       [ 0.04612064,  0.02249181,  0.04885054],
       [-0.02260065,  0.03490208, -0.02947539]], dtype=float32)

In [ ]:
y = model.wv.index_to_key
y

['life',
 'new',
 'young',
 'one',
 'world',
 'family',
 'two',
 'must',
 'man',
 'find',
 'love',
 'years',
 'finds',
 'friends',
 'woman',
 'home',
 'help',
 'time',
 'back',
 'story',
 'school',
 'lives',
 'soon',
 'get',
 'take',
 'father',
 'group',
 'way',
 'mysterious',
 'war',
 'becomes',
 'wife',
 'town',
 'girl',
 'son',
 'three',
 'takes',
 'city',
 'together',
 'mother',
 'become',
 'first',
 'save',
 'daughter',
 'old',
 'friend',
 'make',
 'team',
 'begins',
 'day',
 'death',
 'discovers',
 'high',
 'secret',
 'police',
 'named',
 'best',
 'gets',
 'past',
 'boy',
 'former',
 'people',
 'set',
 'meets',
 'hes',
 'small',
 'go',
 'forced',
 'york',
 'house',
 'however',
 'american',
 'true',
 'order',
 'mission',
 'night',
 'discover',
 'goes',
 'job',
 'living',
 'falls',
 'work',
 'murder',
 'evil',
 'fight',
 'stop',
 'even',
 'tries',
 'agent',
 'battle',
 'journey',
 'parents',
 'comes',
 'men',
 'film',
 'decides',
 'year',
 'earth',
 'turns',
 'killer',
 'turn',
 'c

In [ ]:
import plotly.express as px
fig = px.scatter_3d(store[:100],x=0,y=1,z=2, color=y[:100])
fig.show()